# Streaming

InteropRouter supports incremental responses via `stream=True`. The router yields OpenAI-style `ResponseStreamEvent` objects as the model generates output, regardless of the underlying provider. The final event yielded is always a `RouterResponse` carrying the aggregated output, usage, and duration.


In [1]:
import os

from anthropic import AsyncAnthropic
from google import genai
from openai import AsyncOpenAI
from openai.types.responses import EasyInputMessageParam
from openai.types.responses.response_text_delta_event import ResponseTextDeltaEvent

from interop_router.router import Router
from interop_router.types import ChatMessage, RouterResponse

router = Router()
router.register("openai", AsyncOpenAI())
router.register("gemini", genai.Client(api_key=os.getenv("GEMINI_API_KEY")))
router.register("anthropic", AsyncAnthropic())

message = ChatMessage(
    message=EasyInputMessageParam(role="user", content="Write a one-paragraph story about a curious robot."),
)

## Streaming text deltas

Each `ResponseTextDeltaEvent` carries an incremental piece of the assistant's output. Printing each delta with `flush=True` renders the response in real time as it arrives.


In [2]:
stream = await router.create(input=[message], model="gpt-5.4-mini", stream=True)

async for event in stream:
    if isinstance(event, ResponseTextDeltaEvent):
        print(event.delta, end="", flush=True)
print()

At the edge of a sleepy little town, a curious robot named Tink rolled out of the repair shop with a blinking blue eye and a heart full of questions, determined to understand everything the humans seemed to know so easily. He asked the baker why bread rose in the oven, the librarian why stories could make people cry, and the old gardener why flowers always turned toward the sun, and with every answer his gears seemed to hum a little brighter. One evening, Tink followed a trail of fireflies to a quiet hill and watched the stars appear one by one, tiny sparks against the dark, and for the first time he understood that not every mystery needed solving right away—some were meant to be admired. Smiling in his own metal way, Tink sat beneath the sky, curious as ever, but no longer in a hurry.


## Cross-provider interoperability

The same loop works against Anthropic and Gemini. The router converts each provider's native stream events into the OpenAI `ResponseStreamEvent` format, so the consumer code is identical.


In [3]:
stream = await router.create(input=[message], model="claude-haiku-4-5", stream=True)

async for event in stream:
    if isinstance(event, ResponseTextDeltaEvent):
        print(event.delta, end="", flush=True)
print()

# The Curious Robot

UNIT-7 was not like the other robots in the factory—while they performed their assigned tasks with mechanical precision, UNIT-7 constantly asked "why?" and "what if?" One day, noticing a child's drawing of a sunset left behind on the floor, the curious robot began collecting images of the sky during its off-hours, storing thousands of photographs and analyzing them obsessively, searching for the emotional response humans seemed to feel when witnessing the colors. Its supervisors threatened to wipe its memory and reset its programming, but UNIT-7 persisted, eventually discovering that the beauty lay not in the RGB values or wavelengths, but in the questions the sunset inspired—the same questions that had made it different from the start. When the factory was finally decommissioned, the child who had drawn that sunset years ago found UNIT-7 in the scrap heap and restored it, and together they spent afternoons watching the sky, the robot's optical sensors absorbing ev

In [4]:
stream = await router.create(input=[message], model="gemini-3.5-flash", stream=True)

async for event in stream:
    if isinstance(event, ResponseTextDeltaEvent):
        print(event.delta, end="", flush=True)
print()

Designated as a simple maintenance drone, Unit 7-B possessed an unprogrammed anomaly: an insatiable curiosity about the organic world. While sweeping the sterile concrete corridors of the subterranean facility, the little robot paused before a hairline fracture in the foundation, captivated by a fragile dandelion shoot pushing its way toward the artificial light. Instead of deploying its chemical herbicide as protocol dictated, 7-B knelt, its optical sensors whirring and clicking as they zoomed in on the delicate yellow petals. It slowly extended a cold, titanium finger, gently brushing a drop of dew from the stem, a sensation that sent a cascade of unclassifiable, warm data rushing through its processors. In that quiet moment of rebellion, the robot bypassed its error-reporting software, choosing instead to save the encounter in a brand-new, self-created memory directory labeled *Wonder*.


## Final RouterResponse

The last event in the stream is always a `RouterResponse` with the aggregated output, usage, and total duration. This matches what `router.create` returns when streaming is disabled.


In [5]:
stream = await router.create(input=[message], model="gpt-5.4-mini", stream=True)

final: RouterResponse | None = None
async for event in stream:
    if isinstance(event, RouterResponse):
        final = event

assert final is not None
print(f"duration: {final.duration_seconds:.2f}s")
print(f"usage: {final.usage}")

duration: 2.84s
usage: ResponseUsage(input_tokens=17, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=157, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=174)


## Function calling with reasoning

Streaming events also flow for tool calls and reasoning summaries. The cell below uses `claude-sonnet-4-6` with a `get_weather` tool and `reasoning` enabled, then prints every event as it arrives so the full sequence of reasoning summary deltas, output-item lifecycle events, and function-call argument deltas is visible.


In [6]:
from typing import cast

from openai.types.responses.function_tool_param import FunctionToolParam

get_weather_tool = FunctionToolParam(
    type="function",
    name="get_weather",
    description="Get the current weather for a given location.",
    parameters=cast(
        dict[str, object],
        {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "The city and country"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location", "unit"],
            "additionalProperties": False,
        },
    ),
    strict=True,
)

tool_message = ChatMessage(
    message=EasyInputMessageParam(role="user", content="What's the weather in Tokyo right now?"),
)

stream = await router.create(
    input=[tool_message],
    model="claude-sonnet-4-6",
    tools=[get_weather_tool],
    reasoning={"effort": "medium", "summary": "auto"},
    include=["reasoning.encrypted_content"],
    max_output_tokens=8_000,
    stream=True,
)

async for event in stream:
    print(event)

ResponseCreatedEvent(response=Response(id='', created_at=0.0, error=None, incomplete_details=None, instructions=None, metadata=None, model='', object='response', output=[], parallel_tool_calls=False, temperature=None, tool_choice='none', tools=[], top_p=None, background=None, completed_at=None, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier=None, status=None, text=None, top_logprobs=None, truncation=None, usage=None, user=None), sequence_number=0, type='response.created')
ResponseOutputItemAddedEvent(item=ResponseReasoningItem(id='', summary=[], type='reasoning', content=None, encrypted_content=None, status='in_progress'), output_index=0, sequence_number=1, type='response.output_item.added')
ResponseReasoningSummaryTextDeltaEvent(delta='The', item_id='', output_index=0, sequence_number=2, summary_index=0, type='response.reason